# 20c — CNN on Cats vs Dogs

Corresponds to: Note 20 — CNN Architecture: Convolutional, Pooling, and Fully Connected Layers

**Libraries introduced in this notebook:** `torchvision.datasets.ImageFolder` — loads images from a directory tree where each subdirectory is a class label; used here for the Kaggle Cats vs Dogs dataset.

**What this notebook demonstrates:**
1. Loading real-world photos from disk using ImageFolder
2. Handling variable-size images — resize and centre-crop to a fixed size
3. Training a CNN on binary classification (cat vs dog)
4. The effect of image size on parameter count and training speed
5. Inspecting confidence scores on test images

---

**Dataset setup required before running this notebook.**

Download the Kaggle Cats vs Dogs dataset: https://www.kaggle.com/c/dogs-vs-cats/data

Organise into this directory structure:

```
data/cats_dogs/
    train/
        cat/    <- cat.0.jpg, cat.1.jpg, ...
        dog/    <- dog.0.jpg, dog.1.jpg, ...
    test/
        cat/    <- cat.10000.jpg, ...
        dog/    <- dog.10000.jpg, ...
```

Quick split from the Kaggle `train/` folder:

```python
import shutil
from pathlib import Path

src = Path('data/cats_dogs_raw/train')
for split, start, end in [('train', 0, 10000), ('test', 10000, 12500)]:
    for cls in ['cat', 'dog']:
        dst = Path(f'data/cats_dogs/{split}/{cls}')
        dst.mkdir(parents=True, exist_ok=True)
        for i in range(start, end):
            shutil.copy(src / f'{cls}.{i}.jpg', dst / f'{cls}.{i}.jpg')
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:      {device}')
print(f'PyTorch:     {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')

DATA_ROOT = './data/cats_dogs'
assert os.path.exists(DATA_ROOT), (
    f'Dataset not found at {DATA_ROOT}. See setup instructions in the notebook header.'
)

## Part 1: Load the Dataset

Real photographs — variable size, colour, composition. Resize to 64×64 and centre-crop for a fixed input size.

In [ ]:
IMG_SIZE = 64

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 8),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

train_set = ImageFolder(os.path.join(DATA_ROOT, 'train'), transform=transform)
test_set  = ImageFolder(os.path.join(DATA_ROOT, 'test'),  transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False, num_workers=0)

CLASSES = train_set.classes
print(f'Classes:          {CLASSES}  (index: {train_set.class_to_idx})')
print(f'Training samples: {len(train_set)}')
print(f'Test samples:     {len(test_set)}')
print(f'Image shape:      {train_set[0][0].shape}  (C x H x W)')

## Part 2: Build the CNN

Three conv blocks for 64×64 input — one more than the CIFAR-10 notebook because the images are larger.

In [ ]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3,   32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),  # -> 32x32
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32,  64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),  # -> 16x16
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),  # -> 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(),
            nn.Linear(256, 1),  # binary: BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.classifier(self.block3(self.block2(self.block1(x))))

model = CatDogCNN().to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print('Architecture: 3x64x64 -> 32x32x32 -> 64x16x16 -> 128x8x8 -> flatten -> 256 -> 1')

## Part 3: Train

Binary classification — one output neuron with `BCEWithLogitsLoss`. Prediction: logit > 0 → dog, logit ≤ 0 → cat.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader):
    model.train()
    tl, tc, n = 0.0, 0, 0
    for X, y in loader:
        X = X.to(device)
        y = y.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tl += loss.item()*len(y); tc += ((out>0)==y.bool()).sum().item(); n += len(y)
    return tl/n, tc/n

def eval_epoch(model, loader):
    model.eval()
    tl, tc, n = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y = y.float().unsqueeze(1).to(device)
            out = model(X)
            tl += criterion(out,y).item()*len(y); tc += ((out>0)==y.bool()).sum().item(); n += len(y)
    return tl/n, tc/n

print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Test Loss":<14} {"Test Acc"}')
print('-' * 60)
for epoch in range(1, 16):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    te_loss, te_acc = eval_epoch(model, test_loader)
    print(f'{epoch:<8} {tr_loss:<14.4f} {tr_acc:<14.4f} {te_loss:<14.4f} {te_acc:.4f}')

## Part 4: Confidence Scores

In [ ]:
model.eval()
all_probs, all_labels, all_preds = [], [], []
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(device)).squeeze(1)
        probs  = torch.sigmoid(logits).cpu()
        all_probs.append(probs); all_labels.append(y); all_preds.append((probs > 0.5).long())

all_probs  = torch.cat(all_probs)
all_labels = torch.cat(all_labels)
all_preds  = torch.cat(all_preds)

overall_acc  = (all_preds == all_labels).float().mean().item()
high_conf    = (all_probs > 0.9) | (all_probs < 0.1)
medium_conf  = (all_probs >= 0.4) & (all_probs <= 0.6)

print(f'Overall test accuracy: {overall_acc:.4f}')
print(f'High confidence (>0.9 or <0.1): {high_conf.sum().item()} samples ({high_conf.float().mean().item()*100:.1f}%)')
print(f'Uncertain (0.4 to 0.6):         {medium_conf.sum().item()} samples ({medium_conf.float().mean().item()*100:.1f}%)')
print(f'Accuracy on high-confidence:    {(all_preds[high_conf]==all_labels[high_conf]).float().mean().item():.4f}')

## Summary

What this notebook demonstrated:

1. **ImageFolder** — loads real photographs from a directory tree. Each subdirectory is a class. Handles variable-size images via resize and centre-crop.
2. **Three conv blocks** — deeper than the CIFAR-10 notebook because 64×64 images have more spatial detail to extract before flattening.
3. **Binary classification** — one output neuron with `BCEWithLogitsLoss`. Prediction: logit > 0 → dog, logit ≤ 0 → cat.
4. **Confidence scores** — sigmoid converts logits to probabilities. High-confidence predictions are more accurate than uncertain ones near the decision boundary.
5. **Real-world data** — unlike CIFAR-10 (32×32, clean), Cats vs Dogs has high-resolution, variable-composition photos. The same CNN architecture principles apply but the task is harder.